In [ ]:
!nvidia-smi


In [ ]:
%pip install -q pytorch-lightning==2.6.0 asteroid==0.7.0 flow-matching==1.0.10 einops==0.8.2 pysndfx
!apt-get install -y -q sox


In [ ]:
import os
if not os.path.exists('/content/AD-FlowTSE'):
    !git clone https://github.com/aleXiehta/AD-FlowTSE.git /content/AD-FlowTSE

%cd /content/AD-FlowTSE

init_path = '/content/AD-FlowTSE/models/__init__.py'
with open(init_path, 'w') as f:
    f.write('''# models/__init__.py\n
from .udit.udit import UDiT\n
from .ecapa_tdnn import ECAPA_TDNN\n
\ntry:\n
    from .speakerbeam.td_speakerbeam import TimeDomainSpeakerBeam\n
    from .spex_plus.spex_plus import SpEx_Plus\n
except ImportError:\n
    pass\n
''')
print('Fixed models/__init__.py')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
ckpt_src = '/content/drive/MyDrive/t_predictor_noisy.ckpt'
ckpt_dst = '/content/AD-FlowTSE/t_predictor_noisy.ckpt'

if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, ckpt_dst)
    size_mb = os.path.getsize(ckpt_dst) / 1e6
    print(f'Copied checkpoint: {size_mb:.1f} MB')
else:

    alt = '/content/drive/MyDrive/AD-FlowTSE_checkpoints/t_predictor_noisy.ckpt'
    if os.path.exists(alt):
        shutil.copy(alt, ckpt_dst)
        print(f'Copied from AD-FlowTSE_checkpoints/')
    else:
        print(f'NOT FOUND. Upload t_predictor_noisy.ckpt to Google Drive root!')


In [ ]:
import os, subprocess, sys

local_data = '/content/librimix_data'
librimix_base = os.path.join(local_data, 'Libri2Mix/wav16k/min')

drive_cache = '/content/drive/MyDrive/librimix_data_gate'
if os.path.exists(os.path.join(drive_cache, 'Libri2Mix/wav16k/min/train-100/mix_both')):
    print('Found cached data on Drive! Copying...')
    !cp -r {drive_cache} {local_data}
    print('Done.')
else:
    print('No cache found. Generating LibriMix from scratch...')
    print('This takes ~40-50 minutes (downloading LibriSpeech + WHAM + generating mixtures).')
    print()

    if not os.path.exists('/content/LibriMix'):
        !git clone https://github.com/JorisCos/LibriMix.git /content/LibriMix

    %cd /content/LibriMix
    !bash generate_librimix.sh {local_data}
    %cd /content/AD-FlowTSE

dev_check = os.path.join(librimix_base, 'dev/mix_both')
if not os.path.exists(dev_check):
    print('\nDev split missing, generating explicitly...')
    %cd /content/LibriMix
    !python scripts/create_librimix_from_metadata.py \
        --librispeech_dir {local_data}/LibriSpeech \
        --wham_dir {local_data}/wham_noise \
        --metadata_dir metadata/Libri2Mix \
        --librimix_outdir {local_data}/Libri2Mix \
        --n_src 2 \
        --freqs 16k \
        --modes min \
        --types mix_both mix_clean mix_single
    %cd /content/AD-FlowTSE

print('\nGenerated data:')
for d in ['train-100', 'train-360', 'dev', 'test']:
    path = os.path.join(librimix_base, d)
    if os.path.exists(path):
        mb_dir = os.path.join(path, 'mix_both')
        n = len(os.listdir(mb_dir)) if os.path.exists(mb_dir) else '?'
        print(f'  {d}: {n} files')
    else:
        print(f'  {d}: NOT FOUND')


In [ ]:
%cd /content/AD-FlowTSE

import os, shutil, csv, glob
from collections import defaultdict

librimix_base = '/content/librimix_data/Libri2Mix/wav16k/min'
metadata_dir = os.path.join(librimix_base, 'metadata')
data_out = '/content/AD-FlowTSE/data/wav16k/min'

splits = {
    'train-100': os.path.join(librimix_base, 'train-100'),
    'dev': os.path.join(librimix_base, 'dev'),
    'test': os.path.join(librimix_base, 'test'),
}

for split_name, src_dir in splits.items():
    if not os.path.exists(src_dir):
        print(f'Skipping {split_name} (not found)')
        continue

    dst_dir = os.path.join(data_out, split_name)
    os.makedirs(dst_dir, exist_ok=True)

    for subdir in ['mix_both', 'mix_clean', 'mix_single', 's1', 's2', 'noise']:
        src_sub = os.path.join(src_dir, subdir)
        dst_sub = os.path.join(dst_dir, subdir)

    if os.path.exists(metadata_dir):
        for csv_file in glob.glob(os.path.join(metadata_dir, f'mixture_{split_name}_*.csv')):
            dst_csv = os.path.join(dst_dir, os.path.basename(csv_file))
            if not os.path.exists(dst_csv):
                shutil.copy2(csv_file, dst_csv)
                print(f'Copied {os.path.basename(csv_file)} -> {split_name}/')

    enr_csv = os.path.join(dst_dir, 'mixture2enrollment.csv')
    if os.path.exists(enr_csv):
        print(f'{split_name}/mixture2enrollment.csv already exists')
        continue

    mix_both_csv = None
    for c in glob.glob(os.path.join(dst_dir, 'mixture_*_mix_both.csv')):
        mix_both_csv = c
        break
    if mix_both_csv is None:
        print(f'WARNING: No mix_both CSV for {split_name}')
        continue

    import pandas as pd
    df = pd.read_csv(mix_both_csv)

    spk_sources = defaultdict(list)
    for _, row in df.iterrows():
        mix_id = row['mixture_ID']
        for src_idx in [1, 2]:
            src_path = row[f'source_{src_idx}_path']
            utt_id = mix_id.split('_')[src_idx - 1]
            spk = utt_id.split('-')[0]
            length = row['length']
            spk_sources[spk].append((utt_id, src_path, length))

    rows_out = []
    for _, row in df.iterrows():
        mix_id = row['mixture_ID']
        for src_idx in [1, 2]:
            utt_id = mix_id.split('_')[src_idx - 1]
            spk = utt_id.split('-')[0]
            candidates = [
                (p, l) for uid, p, l in spk_sources[spk] if uid != utt_id
            ]
            if not candidates:
                candidates = [(row[f'source_{src_idx}_path'], row['length'])]
            enr_fields = []
            for p, l in candidates:
                enr_fields.extend([p, str(l)])
            rows_out.append([mix_id, utt_id] + enr_fields)

    with open(enr_csv, 'w', newline='') as f:
        w = csv.writer(f)
        max_cols = max(len(r) for r in rows_out)
        header = ['mixture_ID', 'source_id'] + [
            f'enrollment_{i//2+1}_{"path" if i%2==0 else "length"}'
            for i in range(max_cols - 2)
        ]
        w.writerow(header)
        for r in rows_out:
            w.writerow(r)

    print(f'Generated {enr_csv} ({len(rows_out)} entries)')

print('\nData preparation complete!')


In [ ]:
%cd /content/AD-FlowTSE

gate_model_code = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
from .ecapa_tdnn import ECAPA_TDNN


class SpeakerVerificationGate(nn.Module):
    def __init__(self, C=1024):
        super().__init__()
        self.ecapa_tdnn = ECAPA_TDNN(C=C)
        self.classifier = nn.Sequential(
            nn.Linear(192 * 2 + 1, 128),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, mixture, enrollment, aug=False):
        mix_emb = self.ecapa_tdnn(mixture, aug)
        enr_emb = self.ecapa_tdnn(enrollment, aug)
        mix_norm = F.normalize(mix_emb, dim=-1)
        enr_norm = F.normalize(enr_emb, dim=-1)
        cosine = (mix_norm * enr_norm).sum(dim=-1, keepdim=True)
        features = torch.cat([mix_emb, enr_emb, cosine], dim=-1)
        logits = self.classifier(features).squeeze(-1)
        return logits

    def predict(self, mixture, enrollment):
        with torch.no_grad():
            logits = self.forward(mixture, enrollment, aug=False)
            return torch.sigmoid(logits)
'''

with open('models/speaker_gate.py', 'w') as f:
    f.write(gate_model_code.strip() + '\n')
print('Wrote models/speaker_gate.py')


In [ ]:
%cd /content/AD-FlowTSE

import urllib.request, os

train_script = r'''
import argparse
import os
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import yaml
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping, ModelCheckpoint, LearningRateMonitor,
)
from pytorch_lightning.loggers import TensorBoardLogger
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR
from torch.utils.data import Dataset, DataLoader

from models.speaker_gate import SpeakerVerificationGate


def read_enrollment_csv(csv_path):
    data = defaultdict(dict)
    with open(csv_path, "r") as f:
        f.readline()
        for line in f:
            mix_id, utt_id, *aux = line.strip().split(",")
            aux_it = iter(aux)
            aux = [(p, int(float(l))) for p, l in zip(aux_it, aux_it)]
            data[mix_id][utt_id] = aux
    return data


class SpeakerGateDataset(Dataset):
    def __init__(self, csv_dir, librimix_meta_dir, task="sep_noisy",
                 sample_rate=16000, n_src=2, segment=3, segment_aux=3):
        self.sample_rate = sample_rate
        self.seg_len = int(segment * sample_rate) if segment else None
        self.seg_len_aux = int(segment_aux * sample_rate) if segment_aux else None

        from asteroid.data import LibriMix as AsteroidLibriMix
        base = AsteroidLibriMix(csv_dir, task, sample_rate, n_src, segment)
        self.df = base.df

        enrollment_path = Path(csv_dir) / "mixture2enrollment.csv"
        self.data_aux = read_enrollment_csv(enrollment_path)

        if self.seg_len_aux is not None:
            self.data_aux = {
                m: {u: [(p, l) for p, l in self.data_aux[m][u] if l >= self.seg_len_aux]
                    for u in self.data_aux[m]}
                for m in self.data_aux
            }

        self.pairs = []
        spk_to_enrollment = defaultdict(list)
        for m in self.data_aux:
            for u in self.data_aux[m]:
                if self.data_aux[m][u]:
                    spk = u.split("-")[0]
                    spk_to_enrollment[spk].extend(self.data_aux[m][u])
                    self.pairs.append((m, u))

        self.all_speakers = sorted(spk_to_enrollment.keys())
        self.spk_to_enrollment = {
            s: list(set(spk_to_enrollment[s])) for s in spk_to_enrollment
        }

        self._pair_to_wrong_speakers = {}
        for m, u in self.pairs:
            mix_spks = set()
            for part in m.split("_"):
                mix_spks.add(part.split("-")[0])
            wrong = [s for s in self.all_speakers if s not in mix_spks]
            self._pair_to_wrong_speakers[(m, u)] = wrong

    def __len__(self):
        return len(self.pairs) * 2

    def _load_segment(self, path, seg_len):
        info = sf.info(path)
        total = info.frames
        if seg_len and total > seg_len:
            start = random.randint(0, total - seg_len)
            wav, _ = sf.read(path, dtype="float32", start=start, stop=start + seg_len)
        else:
            wav, _ = sf.read(path, dtype="float32")
        wav = torch.from_numpy(wav)
        if seg_len and wav.shape[-1] < seg_len:
            wav = torch.nn.functional.pad(wav, (0, seg_len - wav.shape[-1]))
        return wav

    def __getitem__(self, idx):
        is_positive = idx % 2 == 0
        pair_idx = idx // 2
        mix_id, utt_id = self.pairs[pair_idx]

        row = self.df[self.df["mixture_ID"] == mix_id].iloc[0]
        mixture_path = row["mixture_path"]
        mix_wav = self._load_segment(mixture_path, self.seg_len)

        if is_positive:
            enr_path, enr_len = random.choice(self.data_aux[mix_id][utt_id])
            enr_wav = self._load_segment(enr_path, self.seg_len_aux)
            label = 1.0
        else:
            wrong_spks = self._pair_to_wrong_speakers[(mix_id, utt_id)]
            if not wrong_spks:
                enr_path, enr_len = random.choice(self.data_aux[mix_id][utt_id])
                enr_wav = self._load_segment(enr_path, self.seg_len_aux)
                label = 1.0
            else:
                wrong_spk = random.choice(wrong_spks)
                wrong_enrollments = [
                    (p, l) for p, l in self.spk_to_enrollment[wrong_spk]
                    if not self.seg_len_aux or l >= self.seg_len_aux
                ]
                if wrong_enrollments:
                    enr_path, enr_len = random.choice(wrong_enrollments)
                else:
                    enr_path, enr_len = random.choice(self.spk_to_enrollment[wrong_spk])
                enr_wav = self._load_segment(enr_path, self.seg_len_aux)
                label = 0.0

        return mix_wav, enr_wav, torch.tensor(label)


class SpeakerGateLightning(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.model = SpeakerVerificationGate(**config["model"])
        self.config = config
        self.save_hyperparameters(config)
        self.freeze_backbone = config["train"].get("freeze_backbone", True)
        self.unfreeze_after = config["train"].get("unfreeze_after_epoch", 10)
        if self.freeze_backbone:
            for param in self.model.ecapa_tdnn.parameters():
                param.requires_grad = False

    def on_train_epoch_start(self):
        if self.freeze_backbone and self.current_epoch >= self.unfreeze_after:
            for param in self.model.ecapa_tdnn.parameters():
                param.requires_grad = True
            self.freeze_backbone = False
            self.print(f"Epoch {self.current_epoch}: Unfroze ECAPA backbone")

    def training_step(self, batch, batch_idx):
        mix_wav, enr_wav, labels = batch
        logits = self.model(mix_wav, enr_wav,
                           aug=self.config["train"].get("spectral_aug", False))
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
        preds = (torch.sigmoid(logits) > 0.5).float()
        acc = (preds == labels).float().mean()
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True,
                 batch_size=mix_wav.size(0))
        self.log("train_acc", acc, on_step=False, on_epoch=True, prog_bar=True,
                 batch_size=mix_wav.size(0))
        return loss

    def validation_step(self, batch, batch_idx):
        mix_wav, enr_wav, labels = batch
        logits = self.model(mix_wav, enr_wav, aug=False)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
        preds = (torch.sigmoid(logits) > 0.5).float()
        acc = (preds == labels).float().mean()
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True,
                 batch_size=mix_wav.size(0))
        self.log("val_acc", acc, on_step=False, on_epoch=True, prog_bar=True,
                 batch_size=mix_wav.size(0))
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=self.config["optim"]["lr"],
            weight_decay=self.config["optim"]["weight_decay"],
        )
        sc = self.config["scheduler"]
        warmup = LambdaLR(optimizer,
            lr_lambda=lambda ep: max(ep / max(sc["warmup_epochs"], 1), 0.1)
            if ep < sc["warmup_epochs"] else 1.0)
        cosine = CosineAnnealingLR(optimizer, T_max=sc["t_max"], eta_min=sc["eta_min"])
        scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                                 milestones=[sc["warmup_epochs"]])
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}


class GateDataModule(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        self.config = config

    def setup(self, stage=None):
        ds = self.config["dataset"]
        self.train_ds = SpeakerGateDataset(
            csv_dir=ds["train_100_dir"], librimix_meta_dir=ds["librimix_meta_dir"],
            task=ds["task"], sample_rate=ds["sample_rate"], n_src=ds["n_src"],
            segment=ds["segment"], segment_aux=ds["segment_aux"])
        self.val_ds = SpeakerGateDataset(
            csv_dir=ds["val_dir"], librimix_meta_dir=ds["librimix_meta_dir"],
            task=ds["task"], sample_rate=ds["sample_rate"], n_src=ds["n_src"],
            segment=ds["segment"], segment_aux=ds["segment_aux"])
        print(f"Train: {len(self.train_ds)} pairs, Val: {len(self.val_ds)} pairs")

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.config["train"]["batch_size"],
                          shuffle=True, num_workers=self.config["train"]["num_workers"],
                          pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.config["train"]["batch_size"],
                          shuffle=False, num_workers=self.config["train"]["num_workers"],
                          pin_memory=True)


def load_ecapa_weights(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    src = {}
    for k, v in ckpt["state_dict"].items():
        if "ecapa_tdnn" in k:
            new_key = k.split("model.")[-1]
            src[new_key] = v
    missing, unexpected = model.load_state_dict(src, strict=False)
    loaded = len(src) - len(unexpected)
    print(f"Loaded {loaded} ECAPA weights from {ckpt_path}")
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    with open(args.config, "r") as f:
        config = yaml.safe_load(f)

    pl.seed_everything(config["seed"])
    torch.set_float32_matmul_precision("medium")
    os.makedirs(config["train"]["log_dir"], exist_ok=True)
    os.makedirs(config["checkpoint"]["dir"], exist_ok=True)

    model = SpeakerGateLightning(config)
    ecapa_ckpt = config["checkpoint"].get("pretrained_ecapa_ckpt")
    if ecapa_ckpt and os.path.exists(ecapa_ckpt):
        model.model = load_ecapa_weights(model.model, ecapa_ckpt)
    else:
        print(f"No pretrained ECAPA found at {ecapa_ckpt}")

    data_module = GateDataModule(config)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=config["early_stopping"]["patience"],
                      verbose=True, mode="min"),
        ModelCheckpoint(dirpath=config["checkpoint"]["dir"],
                        filename="best", save_top_k=1, save_last=True,
                        verbose=True, monitor="val_loss", mode="min"),
        LearningRateMonitor(logging_interval="epoch"),
    ]

    trainer = pl.Trainer(
        max_epochs=config["train"]["num_epochs"],
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        accumulate_grad_batches=config["train"]["accumulation_steps"],
        callbacks=callbacks,
        logger=TensorBoardLogger(save_dir=config["train"]["log_dir"],
                                 name="lightning_logs", version="speaker_gate"),
        log_every_n_steps=50,
        precision=config["train"]["precision"],
        gradient_clip_val=config["train"]["gradient_clip_val"],
    )
    trainer.fit(model, datamodule=data_module)
    print(f"\nDone! Best checkpoint: {config['checkpoint']['dir']}")


if __name__ == "__main__":
    main()
'''

with open('train_speaker_gate.py', 'w') as f:
    f.write(train_script.strip() + '\n')
print('Wrote train_speaker_gate.py')


In [ ]:
%cd /content/AD-FlowTSE
import yaml, os

librimix_base = '/content/AD-FlowTSE/data/wav16k/min'
metadata_dir = '/content/LibriMix/metadata'

config = {
    'train': {
        'batch_size': 32,
        'num_epochs': 20,
        'accumulation_steps': 1,
        'num_workers': 4,
        'precision': '16-mixed',
        'log_dir': 'exp/speaker_gate',
        'gradient_clip_val': 0.5,
        'spectral_aug': True,
        'freeze_backbone': True,
        'unfreeze_after_epoch': 10,
    },
    'model': {'C': 1024},
    'dataset': {
        'enable_train_360': False,
        'enable_train_100': True,
        'train_100_dir': os.path.join(librimix_base, 'train-100'),
        'val_dir': os.path.join(librimix_base, 'dev'),
        'test_dir': os.path.join(librimix_base, 'test'),
        'librimix_meta_dir': metadata_dir,
        'task': 'sep_noisy',
        'n_src': 2,
        'sample_rate': 16000,
        'segment': 3,
        'segment_aux': 3,
        'n_fft': 510,
        'win_length': 510,
        'hop_length': 128,
        'snr_range': [-15, 15],
    },
    'optim': {'optimizer': 'AdamW', 'lr': 0.0003, 'weight_decay': 0.01},
    'scheduler': {'type': 'CosineAnnealingLR', 't_max': 20,
                  'eta_min': 0.000001, 'warmup_epochs': 2},
    'seed': 42,
    'early_stopping': {'enabled': True, 'monitor': 'val_loss',
                       'patience': 5, 'verbose': True, 'mode': 'min', 'delta': 0.0},
    'checkpoint': {'dir': 'exp/speaker_gate/checkpoints', 'ckpt_name': 'best',
                   'save_last': True, 'verbose': True, 'monitor': 'val_loss',
                   'mode': 'min', 'pretrained_ecapa_ckpt': 't_predictor_noisy.ckpt'},
}

os.makedirs('config', exist_ok=True)
with open('config/config_speaker_gate_colab.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Wrote config/config_speaker_gate_colab.yaml')

!python train_speaker_gate.py --config config/config_speaker_gate_colab.yaml


In [ ]:
import shutil, os

drive_out = '/content/drive/MyDrive/AD-FlowTSE_checkpoints'
os.makedirs(drive_out, exist_ok=True)

for src, name in [
    ('exp/speaker_gate/checkpoints/best.ckpt', 'speaker_gate_best.ckpt'),
    ('exp/speaker_gate/checkpoints/last.ckpt', 'speaker_gate_last.ckpt'),
]:
    if os.path.exists(src):
        dst = os.path.join(drive_out, name)
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Saved {name} ({size_mb:.1f} MB)')
    else:
        print(f'Not found: {src}')

print(f'\nCheckpoints saved to: {drive_out}')
print('Download speaker_gate_best.ckpt to your local AD-FlowTSE/ folder.')
